# Polymarket Whale Monitor

This notebook monitors Polymarket for **"Fresh Whale"** activity - detecting when newly created accounts suddenly invest large sums into prediction markets.

## Detection Modes

### 1. Single-Trade Detection (Real-time)
- Detects individual large trades (>$10,000 USD)
- From accounts with few historical trades or young accounts
- Polls every 60 seconds

### 2. Aggregate Position Detection (Periodic)
- Detects **stealth accumulation** patterns
- Finds accounts that build large positions (>$30K) through multiple small trades
- Specifically targets **asymmetric markets** (low-odds bets < 30%)
- Catches "whale" activity that would otherwise evade single-trade detection
- Scans every 30 minutes

## Features
- Real-time monitoring via Polymarket Subgraph (GraphQL)
- Detects large trades from new/inactive accounts
- Detects aggregate position accumulation in low-odds markets
- Discord webhook notifications with color-coded alerts
- Configurable thresholds for both detection modes

## Setup Instructions
1. Run the installation cell below
2. Configure your Discord webhook URL
3. Adjust thresholds as needed (single-trade AND aggregate)
4. Run the monitor!

---

## Step 1: Install Dependencies

In [ ]:
# Install required libraries
!pip install -q requests pandas gql aiohttp python-dateutil

print("Dependencies installed successfully!")

## Step 2: Configuration

### How to Get a Discord Webhook URL:

1. Open Discord and go to your server
2. Right-click on the channel where you want alerts
3. Select **"Edit Channel"**
4. Go to **"Integrations"** in the left sidebar
5. Click **"Webhooks"**
6. Click **"New Webhook"**
7. Give it a name (e.g., "Whale Monitor")
8. Click **"Copy Webhook URL"**
9. Paste it below!

### Alert Types:
- **Green alerts**: Single trades $10K-$50K
- **Orange alerts**: Single trades $50K-$100K
- **Red alerts**: Single trades >$100K
- **Purple alerts**: Aggregate positions (stealth accumulation)

---

In [ ]:
# =============================================================================
# CONFIGURATION - EDIT THESE VALUES
# =============================================================================

# Your Discord Webhook URL (REQUIRED for alerts)
# Leave empty to just log locally without Discord notifications
DISCORD_WEBHOOK_URL = ""  # Paste your webhook URL here

# =============================================================================
# SINGLE-TRADE DETECTION SETTINGS
# =============================================================================

# Minimum trade value to trigger an alert (in USD)
MIN_TRADE_VALUE_USD = 10000  # $10,000 default

# "New Account" Detection Criteria:
# An account is considered "fresh" if EITHER:
#   - It has fewer than MAX_HISTORICAL_TRADES total trades, OR
#   - Its first trade was within the last NEW_ACCOUNT_HOURS hours

MAX_HISTORICAL_TRADES = 5  # Max trades to be considered "new"
NEW_ACCOUNT_HOURS = 72     # Account age threshold (72 hours = 3 days)

# Polling Configuration
POLL_INTERVAL_SECONDS = 60  # Check for new trades every 60 seconds
LOOKBACK_MINUTES = 5        # Look back 5 minutes each poll

# =============================================================================
# AGGREGATE POSITION DETECTION SETTINGS (Stealth Accumulation)
# =============================================================================
# Detects accounts that build large positions through multiple small trades
# in asymmetric (low-odds) markets

ENABLE_AGGREGATE_DETECTION = True  # Set to False to disable

# Minimum aggregate position value to trigger alert (in USD)
AGGREGATE_MIN_POSITION_USD = 30000  # $30,000 default

# Asymmetric market threshold (markets with odds below this are "low-odds")
# 0.30 = 30%, meaning Yes bets at <30% or No bets at <30%
ASYMMETRIC_PRICE_THRESHOLD = 0.30

# Maximum single trade for aggregate detection (stealth pattern)
# If any single trade exceeds this, it's not considered "stealth"
MAX_SINGLE_TRADE_FOR_AGGREGATE = 10000  # $10,000

# How far back to look for aggregate positions (in days)
AGGREGATE_LOOKBACK_DAYS = 14  # 2 weeks

# How often to run aggregate scans (in minutes)
AGGREGATE_SCAN_INTERVAL_MINUTES = 30  # Every 30 minutes

# =============================================================================
# DISPLAY CONFIGURATION
# =============================================================================

print("=" * 60)
print("Configuration loaded!")
print("=" * 60)
print("\nSINGLE-TRADE DETECTION:")
print(f"  - Minimum trade value: ${MIN_TRADE_VALUE_USD:,}")
print(f"  - Max historical trades: {MAX_HISTORICAL_TRADES}")
print(f"  - New account threshold: {NEW_ACCOUNT_HOURS} hours")
print(f"  - Poll interval: {POLL_INTERVAL_SECONDS} seconds")

print("\nAGGREGATE DETECTION:", "ENABLED" if ENABLE_AGGREGATE_DETECTION else "DISABLED")
if ENABLE_AGGREGATE_DETECTION:
    print(f"  - Min aggregate position: ${AGGREGATE_MIN_POSITION_USD:,}")
    print(f"  - Asymmetric threshold: {ASYMMETRIC_PRICE_THRESHOLD:.0%}")
    print(f"  - Max single trade (stealth): ${MAX_SINGLE_TRADE_FOR_AGGREGATE:,}")
    print(f"  - Lookback period: {AGGREGATE_LOOKBACK_DAYS} days")
    print(f"  - Scan interval: {AGGREGATE_SCAN_INTERVAL_MINUTES} minutes")

print("\nDISCORD WEBHOOK:", "Configured" if DISCORD_WEBHOOK_URL else "NOT SET (alerts will only be logged)")

## Step 3: Core Monitoring Code

Run this cell to load all the monitoring functions.

In [ ]:
import os
import sys
import time
import json
import logging
from datetime import datetime, timedelta, timezone
from typing import Optional, Dict, List, Any, Set
from dataclasses import dataclass, field
from IPython.display import display, HTML, clear_output

import requests
import pandas as pd

# =============================================================================
# CONFIGURATION CLASS
# =============================================================================

@dataclass
class Config:
    """Configuration settings for the Whale Monitor."""
    SUBGRAPH_URL: str = (
        "https://api.goldsky.com/api/public/"
        "project_cl6mb8i9h0003e201j6li0diw/subgraphs/polymarket-subgraph/prod/gn"
    )
    DISCORD_WEBHOOK_URL: str = ""
    MIN_TRADE_VALUE_USD: float = 10_000.0
    MAX_HISTORICAL_TRADES: int = 5
    NEW_ACCOUNT_HOURS: int = 72
    POLL_INTERVAL_SECONDS: int = 60
    LOOKBACK_MINUTES: int = 5
    REQUEST_TIMEOUT: int = 30
    MAX_RETRIES: int = 3
    RETRY_DELAY: int = 5
    LOG_LEVEL: str = "INFO"
    # Aggregate detection settings
    AGGREGATE_MIN_POSITION_USD: float = 30_000.0
    ASYMMETRIC_PRICE_THRESHOLD: float = 0.30
    AGGREGATE_LOOKBACK_DAYS: int = 14
    AGGREGATE_SCAN_INTERVAL_MINUTES: int = 30
    MAX_SINGLE_TRADE_FOR_AGGREGATE: float = 10_000.0
    ENABLE_AGGREGATE_DETECTION: bool = True

# =============================================================================
# DATA MODELS
# =============================================================================

@dataclass
class Trade:
    """Represents a trading event on Polymarket."""
    id: str
    user_address: str
    market_id: str
    market_title: str
    outcome: str
    amount: float
    price: float
    value_usd: float
    timestamp: int
    tx_hash: str

    @property
    def formatted_time(self) -> str:
        return datetime.fromtimestamp(
            self.timestamp, tz=timezone.utc
        ).strftime("%Y-%m-%d %H:%M:%S UTC")


@dataclass
class AccountProfile:
    """Profile information for a Polymarket account."""
    address: str
    total_trades: int
    first_trade_timestamp: Optional[int]
    total_volume_usd: float
    markets_traded: int

    @property
    def account_age_hours(self) -> Optional[float]:
        if not self.first_trade_timestamp:
            return None
        age_seconds = time.time() - self.first_trade_timestamp
        return age_seconds / 3600

    def is_fresh_whale(self, config: Config) -> bool:
        if self.total_trades < config.MAX_HISTORICAL_TRADES:
            return True
        if self.account_age_hours is not None:
            if self.account_age_hours < config.NEW_ACCOUNT_HOURS:
                return True
        return False


@dataclass
class MarketPosition:
    """Aggregated position in a single market."""
    market_id: str
    market_title: str
    outcome: str
    total_shares: float
    average_price: float
    total_invested_usd: float
    trade_count: int
    first_trade_timestamp: int
    last_trade_timestamp: int
    max_single_trade_usd: float

    @property
    def is_asymmetric(self) -> bool:
        return self.average_price < 0.30

    @property
    def time_span_hours(self) -> float:
        if self.first_trade_timestamp and self.last_trade_timestamp:
            return (self.last_trade_timestamp - self.first_trade_timestamp) / 3600
        return 0


@dataclass
class FreshWhaleAlert:
    """Alert data for a Fresh Whale detection."""
    trade: Trade
    profile: AccountProfile
    detection_reason: str
    alert_time: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

    def to_discord_embed(self) -> Dict[str, Any]:
        if self.trade.value_usd >= 100_000:
            color = 0xFF0000
        elif self.trade.value_usd >= 50_000:
            color = 0xFFA500
        else:
            color = 0x00FF00

        profile_url = f"https://polymarket.com/profile/{self.trade.user_address}"

        if self.profile.account_age_hours is not None:
            if self.profile.account_age_hours < 1:
                age_str = f"{int(self.profile.account_age_hours * 60)} minutes"
            elif self.profile.account_age_hours < 24:
                age_str = f"{self.profile.account_age_hours:.1f} hours"
            else:
                age_str = f"{self.profile.account_age_hours / 24:.1f} days"
        else:
            age_str = "Unknown"

        return {
            "title": "Fresh Whale Detected!",
            "description": "A new account just made a large trade on Polymarket.",
            "color": color,
            "fields": [
                {"name": "Market", "value": self.trade.market_title[:256], "inline": False},
                {"name": "Position", "value": f"**{self.trade.outcome}** @ ${self.trade.price:.3f}", "inline": True},
                {"name": "Amount Invested", "value": f"**${self.trade.value_usd:,.2f}**", "inline": True},
                {"name": "Shares Purchased", "value": f"{self.trade.amount:,.2f}", "inline": True},
                {"name": "Account Age", "value": age_str, "inline": True},
                {"name": "Total Prior Trades", "value": str(self.profile.total_trades), "inline": True},
                {"name": "Detection Reason", "value": self.detection_reason, "inline": True},
                {"name": "Wallet Address", "value": f"`{self.trade.user_address[:10]}...{self.trade.user_address[-8:]}`", "inline": False}
            ],
            "timestamp": self.alert_time.isoformat(),
            "footer": {"text": "Polymarket Whale Monitor"},
            "url": profile_url
        }


@dataclass
class AggregateWhaleAlert:
    """Alert data for aggregate position detection (stealth accumulation)."""
    address: str
    position: MarketPosition
    profile: AccountProfile
    detection_reason: str
    alert_time: datetime = field(default_factory=lambda: datetime.now(timezone.utc))

    def to_discord_embed(self) -> Dict[str, Any]:
        if self.position.total_invested_usd >= 100_000:
            color = 0x8B008B  # Dark magenta
        elif self.position.total_invested_usd >= 50_000:
            color = 0x9B59B6  # Purple
        else:
            color = 0xAA88FF  # Light purple

        profile_url = f"https://polymarket.com/profile/{self.address}"

        if self.profile.account_age_hours is not None:
            if self.profile.account_age_hours < 1:
                age_str = f"{int(self.profile.account_age_hours * 60)} minutes"
            elif self.profile.account_age_hours < 24:
                age_str = f"{self.profile.account_age_hours:.1f} hours"
            else:
                age_str = f"{self.profile.account_age_hours / 24:.1f} days"
        else:
            age_str = "Unknown"

        if self.position.time_span_hours < 1:
            span_str = f"{int(self.position.time_span_hours * 60)} minutes"
        elif self.position.time_span_hours < 24:
            span_str = f"{self.position.time_span_hours:.1f} hours"
        else:
            span_str = f"{self.position.time_span_hours / 24:.1f} days"

        return {
            "title": "Aggregate Position Detected!",
            "description": "Account accumulated a large position through multiple small trades in a low-odds market.",
            "color": color,
            "fields": [
                {"name": "Market", "value": self.position.market_title[:256], "inline": False},
                {"name": "Position", "value": f"**{self.position.outcome}** @ avg {self.position.average_price:.1%}", "inline": True},
                {"name": "Total Invested", "value": f"**${self.position.total_invested_usd:,.2f}**", "inline": True},
                {"name": "Total Shares", "value": f"{self.position.total_shares:,.2f}", "inline": True},
                {"name": "Number of Trades", "value": str(self.position.trade_count), "inline": True},
                {"name": "Largest Single Trade", "value": f"${self.position.max_single_trade_usd:,.2f}", "inline": True},
                {"name": "Accumulation Period", "value": span_str, "inline": True},
                {"name": "Account Age", "value": age_str, "inline": True},
                {"name": "Total Prior Trades", "value": str(self.profile.total_trades), "inline": True},
                {"name": "Detection Reason", "value": self.detection_reason, "inline": False},
                {"name": "Wallet Address", "value": f"`{self.address[:10]}...{self.address[-8:]}`", "inline": False}
            ],
            "timestamp": self.alert_time.isoformat(),
            "footer": {"text": "Polymarket Whale Monitor - Aggregate Detection"},
            "url": profile_url
        }

# =============================================================================
# GRAPHQL CLIENT
# =============================================================================

class PolymarketSubgraph:
    """Client for interacting with the Polymarket Subgraph."""

    def __init__(self, config: Config):
        self.config = config
        self.session = requests.Session()

    def _execute_query(self, query: str, variables: Optional[Dict] = None) -> Optional[Dict[str, Any]]:
        payload = {"query": query}
        if variables:
            payload["variables"] = variables

        for attempt in range(self.config.MAX_RETRIES):
            try:
                response = self.session.post(
                    self.config.SUBGRAPH_URL,
                    json=payload,
                    timeout=self.config.REQUEST_TIMEOUT,
                    headers={"Content-Type": "application/json"}
                )
                response.raise_for_status()
                result = response.json()
                if "errors" in result:
                    print(f"GraphQL errors: {result['errors']}")
                    return None
                return result.get("data")
            except requests.exceptions.RequestException as e:
                print(f"Request failed (attempt {attempt + 1}): {e}")
                if attempt < self.config.MAX_RETRIES - 1:
                    time.sleep(self.config.RETRY_DELAY)
        return None

    def get_recent_trades(self, since_timestamp: int, min_value_usd: float = 0, first: int = 100) -> List[Trade]:
        """Fetch recent trades from the subgraph."""
        query = """
        query GetRecentTrades($since: BigInt!, $first: Int!) {
            trades(first: $first, orderBy: timestamp, orderDirection: desc, where: { timestamp_gte: $since }) {
                id
                user { id }
                market { id question }
                outcome
                amount
                price
                timestamp
                transactionHash
            }
        }
        """

        position_query = """
        query GetRecentPositions($since: BigInt!, $first: Int!) {
            fpmmTrades(first: $first, orderBy: creationTimestamp, orderDirection: desc, where: { creationTimestamp_gte: $since }) {
                id
                creator { id }
                fpmm { id question }
                outcomeIndex
                collateralAmount
                outcomeTokensAmount
                creationTimestamp
                transactionHash
            }
        }
        """

        variables = {"since": str(since_timestamp), "first": first}
        data = self._execute_query(query, variables)
        trades = []

        if data and "trades" in data:
            for t in data["trades"]:
                try:
                    amount = float(t.get("amount", 0))
                    price = float(t.get("price", 0))
                    value_usd = amount * price
                    if value_usd < min_value_usd:
                        continue
                    trade = Trade(
                        id=t["id"], user_address=t["user"]["id"], market_id=t["market"]["id"],
                        market_title=t["market"].get("question", "Unknown Market"),
                        outcome=t.get("outcome", "Unknown"), amount=amount, price=price,
                        value_usd=value_usd, timestamp=int(t["timestamp"]), tx_hash=t.get("transactionHash", "")
                    )
                    trades.append(trade)
                except (KeyError, ValueError, TypeError):
                    continue

        if not trades:
            data = self._execute_query(position_query, variables)
            if data and "fpmmTrades" in data:
                for t in data["fpmmTrades"]:
                    try:
                        collateral = float(t.get("collateralAmount", 0)) / 1e6
                        outcome_tokens = float(t.get("outcomeTokensAmount", 0)) / 1e18
                        price = collateral / outcome_tokens if outcome_tokens > 0 else 0
                        if collateral < min_value_usd:
                            continue
                        outcome = "Yes" if int(t.get("outcomeIndex", 0)) == 0 else "No"
                        trade = Trade(
                            id=t["id"], user_address=t["creator"]["id"], market_id=t["fpmm"]["id"],
                            market_title=t["fpmm"].get("question", "Unknown Market"),
                            outcome=outcome, amount=outcome_tokens, price=price,
                            value_usd=collateral, timestamp=int(t["creationTimestamp"]), tx_hash=t.get("transactionHash", "")
                        )
                        trades.append(trade)
                    except (KeyError, ValueError, TypeError):
                        continue
        return trades

    def get_account_profile(self, address: str) -> Optional[AccountProfile]:
        """Fetch account history and profile information."""
        query = """
        query GetAccountProfile($address: String!) {
            user(id: $address) {
                id
                trades(first: 1000, orderBy: timestamp, orderDirection: asc) {
                    id timestamp amount price
                }
            }
        }
        """

        alt_query = """
        query GetAccountProfile($address: String!) {
            account(id: $address) {
                id
                fpmmTrades(first: 1000, orderBy: creationTimestamp, orderDirection: asc) {
                    id creationTimestamp collateralAmount fpmm { id }
                }
            }
        }
        """

        variables = {"address": address.lower()}
        data = self._execute_query(query, variables)

        if data and data.get("user"):
            user = data["user"]
            trades = user.get("trades", [])
            return AccountProfile(
                address=address, total_trades=len(trades),
                first_trade_timestamp=int(trades[0]["timestamp"]) if trades else None,
                total_volume_usd=sum(float(t.get("amount", 0)) * float(t.get("price", 0)) for t in trades),
                markets_traded=0
            )

        data = self._execute_query(alt_query, variables)
        if data and data.get("account"):
            account = data["account"]
            trades = account.get("fpmmTrades", [])
            markets = set(t.get("fpmm", {}).get("id", "") for t in trades)
            return AccountProfile(
                address=address, total_trades=len(trades),
                first_trade_timestamp=int(trades[0]["creationTimestamp"]) if trades else None,
                total_volume_usd=sum(float(t.get("collateralAmount", 0)) / 1e6 for t in trades),
                markets_traded=len(markets)
            )

        return AccountProfile(address=address, total_trades=0, first_trade_timestamp=None, total_volume_usd=0, markets_traded=0)

    def get_account_positions(self, address: str, since_timestamp: int) -> Dict[str, MarketPosition]:
        """Fetch all trades for an account since timestamp, aggregated by market."""
        query = """
        query GetAccountRecentTrades($address: String!, $since: BigInt!) {
            user(id: $address) {
                trades(first: 1000, where: { timestamp_gte: $since }, orderBy: timestamp, orderDirection: asc) {
                    id market { id question } outcome amount price timestamp
                }
            }
        }
        """

        alt_query = """
        query GetAccountRecentTrades($address: String!, $since: BigInt!) {
            account(id: $address) {
                fpmmTrades(first: 1000, where: { creationTimestamp_gte: $since }, orderBy: creationTimestamp, orderDirection: asc) {
                    id fpmm { id question } outcomeIndex collateralAmount outcomeTokensAmount creationTimestamp
                }
            }
        }
        """

        variables = {"address": address.lower(), "since": str(since_timestamp)}
        data = self._execute_query(query, variables)
        positions: Dict[str, MarketPosition] = {}

        if data and data.get("user"):
            trades = data["user"].get("trades", [])
            positions = self._aggregate_trades_to_positions(trades, "primary")

        if not positions:
            data = self._execute_query(alt_query, variables)
            if data and data.get("account"):
                trades = data["account"].get("fpmmTrades", [])
                positions = self._aggregate_trades_to_positions(trades, "fpmm")

        return positions

    def _aggregate_trades_to_positions(self, trades: List[Dict], schema_type: str) -> Dict[str, MarketPosition]:
        """Aggregate individual trades into market positions."""
        position_data: Dict[str, Dict] = {}

        for t in trades:
            try:
                if schema_type == "primary":
                    market_id = t["market"]["id"]
                    market_title = t["market"].get("question", "Unknown Market")
                    outcome = t.get("outcome", "Unknown")
                    amount = float(t.get("amount", 0))
                    price = float(t.get("price", 0))
                    value_usd = amount * price
                    timestamp = int(t["timestamp"])
                else:
                    market_id = t["fpmm"]["id"]
                    market_title = t["fpmm"].get("question", "Unknown Market")
                    outcome = "Yes" if int(t.get("outcomeIndex", 0)) == 0 else "No"
                    collateral = float(t.get("collateralAmount", 0)) / 1e6
                    outcome_tokens = float(t.get("outcomeTokensAmount", 0)) / 1e18
                    price = collateral / outcome_tokens if outcome_tokens > 0 else 0
                    amount = outcome_tokens
                    value_usd = collateral
                    timestamp = int(t["creationTimestamp"])

                key = f"{market_id}:{outcome}"
                if key not in position_data:
                    position_data[key] = {
                        "market_id": market_id, "market_title": market_title, "outcome": outcome,
                        "total_shares": 0, "total_invested_usd": 0, "trade_count": 0,
                        "first_trade_timestamp": timestamp, "last_trade_timestamp": timestamp,
                        "max_single_trade_usd": 0, "prices": []
                    }

                pos = position_data[key]
                pos["total_shares"] += amount
                pos["total_invested_usd"] += value_usd
                pos["trade_count"] += 1
                pos["last_trade_timestamp"] = max(pos["last_trade_timestamp"], timestamp)
                pos["first_trade_timestamp"] = min(pos["first_trade_timestamp"], timestamp)
                pos["max_single_trade_usd"] = max(pos["max_single_trade_usd"], value_usd)
                pos["prices"].append((value_usd, price))
            except (KeyError, ValueError, TypeError):
                continue

        positions: Dict[str, MarketPosition] = {}
        for key, pos in position_data.items():
            total_value = sum(p[0] for p in pos["prices"])
            avg_price = sum(p[0] * p[1] for p in pos["prices"]) / total_value if total_value > 0 else 0
            positions[key] = MarketPosition(
                market_id=pos["market_id"], market_title=pos["market_title"], outcome=pos["outcome"],
                total_shares=pos["total_shares"], average_price=avg_price,
                total_invested_usd=pos["total_invested_usd"], trade_count=pos["trade_count"],
                first_trade_timestamp=pos["first_trade_timestamp"], last_trade_timestamp=pos["last_trade_timestamp"],
                max_single_trade_usd=pos["max_single_trade_usd"]
            )
        return positions

    def get_recent_active_accounts(self, since_timestamp: int, min_trades: int = 2, first: int = 1000) -> List[str]:
        """Get list of accounts that have been active since timestamp."""
        query = """
        query GetRecentTrades($since: BigInt!, $first: Int!) {
            trades(first: $first, where: { timestamp_gte: $since }, orderBy: timestamp, orderDirection: desc) {
                user { id }
            }
        }
        """

        alt_query = """
        query GetRecentTrades($since: BigInt!, $first: Int!) {
            fpmmTrades(first: $first, where: { creationTimestamp_gte: $since }, orderBy: creationTimestamp, orderDirection: desc) {
                creator { id }
            }
        }
        """

        variables = {"since": str(since_timestamp), "first": first}
        data = self._execute_query(query, variables)
        account_trades: Dict[str, int] = {}

        if data and "trades" in data:
            for t in data["trades"]:
                addr = t.get("user", {}).get("id", "")
                if addr:
                    account_trades[addr] = account_trades.get(addr, 0) + 1

        if not account_trades:
            data = self._execute_query(alt_query, variables)
            if data and "fpmmTrades" in data:
                for t in data["fpmmTrades"]:
                    addr = t.get("creator", {}).get("id", "")
                    if addr:
                        account_trades[addr] = account_trades.get(addr, 0) + 1

        return [addr for addr, count in account_trades.items() if count >= min_trades]

# =============================================================================
# DISCORD NOTIFIER
# =============================================================================

class DiscordNotifier:
    """Sends alerts to Discord via webhook."""

    def __init__(self, config: Config):
        self.config = config
        self.session = requests.Session()

    def send_alert(self, alert: FreshWhaleAlert) -> bool:
        if not self.config.DISCORD_WEBHOOK_URL:
            return False
        payload = {"username": "Polymarket Whale Monitor", "embeds": [alert.to_discord_embed()]}
        try:
            response = self.session.post(self.config.DISCORD_WEBHOOK_URL, json=payload, timeout=self.config.REQUEST_TIMEOUT)
            return response.status_code == 204
        except requests.exceptions.RequestException:
            return False

    def send_aggregate_alert(self, alert: AggregateWhaleAlert) -> bool:
        if not self.config.DISCORD_WEBHOOK_URL:
            return False
        payload = {"username": "Polymarket Whale Monitor", "embeds": [alert.to_discord_embed()]}
        try:
            response = self.session.post(self.config.DISCORD_WEBHOOK_URL, json=payload, timeout=self.config.REQUEST_TIMEOUT)
            return response.status_code == 204
        except requests.exceptions.RequestException:
            return False

    def send_startup_notification(self) -> bool:
        if not self.config.DISCORD_WEBHOOK_URL:
            return False

        desc_lines = [
            f"Monitoring for trades above **${self.config.MIN_TRADE_VALUE_USD:,.0f}**",
            f"Polling every **{self.config.POLL_INTERVAL_SECONDS}** seconds"
        ]
        if self.config.ENABLE_AGGREGATE_DETECTION:
            desc_lines.extend([
                "", "**Aggregate Detection Enabled:**",
                f"• Min position: **${self.config.AGGREGATE_MIN_POSITION_USD:,.0f}**",
                f"• Asymmetric threshold: **{self.config.ASYMMETRIC_PRICE_THRESHOLD:.0%}**",
                f"• Lookback: **{self.config.AGGREGATE_LOOKBACK_DAYS}** days"
            ])

        payload = {
            "username": "Polymarket Whale Monitor",
            "embeds": [{
                "title": "Whale Monitor Started",
                "description": "\n".join(desc_lines),
                "color": 0x0099FF,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "footer": {"text": "Polymarket Whale Monitor"}
            }]
        }
        try:
            response = self.session.post(self.config.DISCORD_WEBHOOK_URL, json=payload, timeout=self.config.REQUEST_TIMEOUT)
            return response.status_code == 204
        except requests.exceptions.RequestException:
            return False

# =============================================================================
# WHALE MONITOR
# =============================================================================

class WhaleMonitor:
    """Main monitoring orchestrator."""

    def __init__(self, config: Config):
        self.config = config
        self.subgraph = PolymarketSubgraph(config)
        self.notifier = DiscordNotifier(config)
        self.processed_trade_ids: Set[str] = set()
        self.alerted_aggregate_positions: Set[str] = set()
        self.last_aggregate_scan: float = 0
        self.alerts_history: List[FreshWhaleAlert] = []
        self.aggregate_alerts_history: List[AggregateWhaleAlert] = []
        self.stats = {
            "total_trades_scanned": 0, "large_trades_found": 0,
            "fresh_whales_detected": 0, "aggregate_whales_detected": 0,
            "alerts_sent": 0, "aggregate_alerts_sent": 0, "start_time": None
        }

    def process_trade(self, trade: Trade) -> Optional[FreshWhaleAlert]:
        if trade.id in self.processed_trade_ids:
            return None
        self.processed_trade_ids.add(trade.id)
        self.stats["total_trades_scanned"] += 1

        if trade.value_usd < self.config.MIN_TRADE_VALUE_USD:
            return None

        self.stats["large_trades_found"] += 1
        profile = self.subgraph.get_account_profile(trade.user_address)
        if profile is None or not profile.is_fresh_whale(self.config):
            return None

        if profile.total_trades < self.config.MAX_HISTORICAL_TRADES:
            reason = f"Only {profile.total_trades} prior trades"
        elif profile.account_age_hours is not None and profile.account_age_hours < self.config.NEW_ACCOUNT_HOURS:
            reason = f"Account only {profile.account_age_hours:.1f} hours old"
        else:
            reason = "New account pattern detected"

        self.stats["fresh_whales_detected"] += 1
        alert = FreshWhaleAlert(trade=trade, profile=profile, detection_reason=reason)
        self.alerts_history.append(alert)
        return alert

    def check_aggregate_whale(self, address: str, positions: Dict[str, MarketPosition]) -> List[AggregateWhaleAlert]:
        """Check if account has accumulated large positions through small trades."""
        alerts = []
        for key, position in positions.items():
            alert_key = f"{address}:{key}"
            if alert_key in self.alerted_aggregate_positions:
                continue
            if position.total_invested_usd < self.config.AGGREGATE_MIN_POSITION_USD:
                continue
            if position.average_price >= self.config.ASYMMETRIC_PRICE_THRESHOLD:
                continue
            if position.max_single_trade_usd >= self.config.MAX_SINGLE_TRADE_FOR_AGGREGATE:
                continue
            if position.trade_count < 2:
                continue

            profile = self.subgraph.get_account_profile(address)
            if profile is None or not profile.is_fresh_whale(self.config):
                continue

            reason = (f"Accumulated ${position.total_invested_usd:,.0f} through "
                     f"{position.trade_count} trades at avg {position.average_price:.1%} "
                     f"(max single: ${position.max_single_trade_usd:,.0f})")

            alert = AggregateWhaleAlert(address=address, position=position, profile=profile, detection_reason=reason)
            alerts.append(alert)
            self.alerted_aggregate_positions.add(alert_key)
            self.aggregate_alerts_history.append(alert)
            self.stats["aggregate_whales_detected"] += 1

        return alerts

    def run_aggregate_scan(self) -> List[AggregateWhaleAlert]:
        """Scan for aggregate whale patterns."""
        if not self.config.ENABLE_AGGREGATE_DETECTION:
            return []

        alerts = []
        lookback_seconds = self.config.AGGREGATE_LOOKBACK_DAYS * 24 * 3600
        since_timestamp = int(time.time()) - lookback_seconds

        active_accounts = self.subgraph.get_recent_active_accounts(since_timestamp=since_timestamp, min_trades=2)

        for address in active_accounts:
            try:
                positions = self.subgraph.get_account_positions(address=address, since_timestamp=since_timestamp)
                account_alerts = self.check_aggregate_whale(address, positions)
                for alert in account_alerts:
                    alerts.append(alert)
                    if self.notifier.send_aggregate_alert(alert):
                        self.stats["aggregate_alerts_sent"] += 1
            except Exception:
                continue

        return alerts

    def run_single_poll(self) -> List[FreshWhaleAlert]:
        alerts = []
        lookback_seconds = self.config.LOOKBACK_MINUTES * 60
        since_timestamp = int(time.time()) - lookback_seconds

        trades = self.subgraph.get_recent_trades(since_timestamp=since_timestamp, min_value_usd=self.config.MIN_TRADE_VALUE_USD)
        for trade in trades:
            alert = self.process_trade(trade)
            if alert:
                alerts.append(alert)
                if self.notifier.send_alert(alert):
                    self.stats["alerts_sent"] += 1
        return alerts

    def display_status(self, iteration: int, alerts: List[FreshWhaleAlert], aggregate_alerts: List[AggregateWhaleAlert] = None):
        """Display status in Colab-friendly format."""
        clear_output(wait=True)
        runtime = datetime.now(timezone.utc) - self.stats["start_time"]

        html = f"""
        <div style="font-family: monospace; background: #1e1e1e; color: #00ff00; padding: 20px; border-radius: 10px;">
            <h2 style="color: #00ff00;">Polymarket Whale Monitor</h2>
            <hr style="border-color: #00ff00;">
            <p><strong>Status:</strong> Running (Poll #{iteration})</p>
            <p><strong>Runtime:</strong> {runtime}</p>
            <p><strong>Last Check:</strong> {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC')}</p>
            <hr style="border-color: #00ff00;">
            <h3>Single-Trade Statistics</h3>
            <ul>
                <li>Trades Scanned: {self.stats['total_trades_scanned']}</li>
                <li>Large Trades Found: {self.stats['large_trades_found']}</li>
                <li>Fresh Whales Detected: {self.stats['fresh_whales_detected']}</li>
                <li>Alerts Sent: {self.stats['alerts_sent']}</li>
            </ul>
        """

        if self.config.ENABLE_AGGREGATE_DETECTION:
            html += f"""
            <h3 style="color: #9B59B6;">Aggregate Detection Statistics</h3>
            <ul style="color: #9B59B6;">
                <li>Aggregate Whales Detected: {self.stats['aggregate_whales_detected']}</li>
                <li>Aggregate Alerts Sent: {self.stats['aggregate_alerts_sent']}</li>
            </ul>
            """

        if self.alerts_history:
            html += "<hr style='border-color: #ff6600;'><h3 style='color: #ff6600;'>Recent Single-Trade Alerts</h3>"
            for alert in self.alerts_history[-3:]:
                html += f"""
                <div style="background: #2d2d2d; padding: 10px; margin: 10px 0; border-left: 4px solid #ff6600;">
                    <strong style="color: #ff6600;">${alert.trade.value_usd:,.2f}</strong> on
                    <em>{alert.trade.market_title[:50]}...</em><br>
                    <small>Reason: {alert.detection_reason} | Time: {alert.alert_time.strftime('%H:%M:%S')}</small>
                </div>
                """

        if self.aggregate_alerts_history:
            html += "<hr style='border-color: #9B59B6;'><h3 style='color: #9B59B6;'>Recent Aggregate Alerts</h3>"
            for alert in self.aggregate_alerts_history[-3:]:
                html += f"""
                <div style="background: #2d2d2d; padding: 10px; margin: 10px 0; border-left: 4px solid #9B59B6;">
                    <strong style="color: #9B59B6;">${alert.position.total_invested_usd:,.2f}</strong> aggregated in
                    <em>{alert.position.market_title[:50]}...</em><br>
                    <small>{alert.position.trade_count} trades @ avg {alert.position.average_price:.1%} | Time: {alert.alert_time.strftime('%H:%M:%S')}</small>
                </div>
                """

        html += "</div>"
        display(HTML(html))

    def run_colab(self, max_iterations: Optional[int] = None):
        """Run monitor with Colab-friendly output."""
        self.stats["start_time"] = datetime.now(timezone.utc)

        print("Starting Polymarket Whale Monitor...")
        print(f"Single-trade threshold: ${self.config.MIN_TRADE_VALUE_USD:,}")
        if self.config.ENABLE_AGGREGATE_DETECTION:
            print(f"Aggregate detection: ENABLED (min ${self.config.AGGREGATE_MIN_POSITION_USD:,} @ <{self.config.ASYMMETRIC_PRICE_THRESHOLD:.0%})")
        print(f"Poll interval: {self.config.POLL_INTERVAL_SECONDS} seconds")

        self.notifier.send_startup_notification()

        iteration = 0
        try:
            while True:
                iteration += 1
                if max_iterations and iteration > max_iterations:
                    print(f"Reached max iterations ({max_iterations}), stopping")
                    break

                try:
                    alerts = self.run_single_poll()
                    aggregate_alerts = []

                    # Check if it's time for aggregate scan
                    if self.config.ENABLE_AGGREGATE_DETECTION:
                        now = time.time()
                        scan_interval = self.config.AGGREGATE_SCAN_INTERVAL_MINUTES * 60
                        if now - self.last_aggregate_scan >= scan_interval:
                            aggregate_alerts = self.run_aggregate_scan()
                            self.last_aggregate_scan = now

                    self.display_status(iteration, alerts, aggregate_alerts)
                except Exception as e:
                    print(f"Error in poll cycle: {e}")

                if max_iterations is None or iteration < max_iterations:
                    time.sleep(self.config.POLL_INTERVAL_SECONDS)

        except KeyboardInterrupt:
            print("\nMonitor stopped by user.")

print("Core monitoring code loaded successfully!")
print("  - Single-trade detection: Ready")
print("  - Aggregate position detection: Ready")

## Step 4: Test the Connection

Let's verify the subgraph connection is working before starting the monitor.

In [ ]:
# Test the subgraph connection
test_config = Config()
test_subgraph = PolymarketSubgraph(test_config)

print("Testing Polymarket Subgraph connection...")

# Try to fetch some recent trades
since_ts = int(time.time()) - 3600  # Last hour
trades = test_subgraph.get_recent_trades(since_ts, min_value_usd=0, first=10)

if trades:
    print(f"Connection successful! Found {len(trades)} recent trades.")
    print("\nSample trade:")
    t = trades[0]
    print(f"  Market: {t.market_title[:60]}...")
    print(f"  Value: ${t.value_usd:,.2f}")
    print(f"  Outcome: {t.outcome}")
else:
    print("No trades found in the last hour, but connection may still be working.")
    print("The subgraph schema may differ from expected - check for updates.")

## Step 5: Start the Whale Monitor

Run the cell below to start monitoring. The display will update every polling cycle.

**To stop the monitor:** Click the "Stop" button in the Colab toolbar, or press `Ctrl+M I` to interrupt.

---

In [ ]:
# =============================================================================
# START THE WHALE MONITOR
# =============================================================================

# Build configuration from the settings above
config = Config(
    DISCORD_WEBHOOK_URL=DISCORD_WEBHOOK_URL,
    # Single-trade detection settings
    MIN_TRADE_VALUE_USD=MIN_TRADE_VALUE_USD,
    MAX_HISTORICAL_TRADES=MAX_HISTORICAL_TRADES,
    NEW_ACCOUNT_HOURS=NEW_ACCOUNT_HOURS,
    POLL_INTERVAL_SECONDS=POLL_INTERVAL_SECONDS,
    LOOKBACK_MINUTES=LOOKBACK_MINUTES,
    # Aggregate detection settings
    ENABLE_AGGREGATE_DETECTION=ENABLE_AGGREGATE_DETECTION,
    AGGREGATE_MIN_POSITION_USD=AGGREGATE_MIN_POSITION_USD,
    ASYMMETRIC_PRICE_THRESHOLD=ASYMMETRIC_PRICE_THRESHOLD,
    MAX_SINGLE_TRADE_FOR_AGGREGATE=MAX_SINGLE_TRADE_FOR_AGGREGATE,
    AGGREGATE_LOOKBACK_DAYS=AGGREGATE_LOOKBACK_DAYS,
    AGGREGATE_SCAN_INTERVAL_MINUTES=AGGREGATE_SCAN_INTERVAL_MINUTES
)

# Create and run the monitor
monitor = WhaleMonitor(config)

# Run indefinitely (or set max_iterations for testing)
# For testing, uncomment the line below:
# monitor.run_colab(max_iterations=5)

# For production, run indefinitely:
monitor.run_colab()

---

## Viewing Alert History

After stopping the monitor, run this cell to see all detected whales (both single-trade and aggregate):

In [ ]:
# View all detected Fresh Whales
if 'monitor' in dir():
    # Single-trade alerts
    if monitor.alerts_history:
        print("=" * 60)
        print("SINGLE-TRADE WHALE ALERTS")
        print("=" * 60)
        df_single = pd.DataFrame([
            {
                "Time": alert.alert_time.strftime("%Y-%m-%d %H:%M:%S"),
                "Value (USD)": f"${alert.trade.value_usd:,.2f}",
                "Market": alert.trade.market_title[:50] + "...",
                "Outcome": alert.trade.outcome,
                "Price": f"${alert.trade.price:.3f}",
                "Account Age (hrs)": f"{alert.profile.account_age_hours:.1f}" if alert.profile.account_age_hours else "N/A",
                "Prior Trades": alert.profile.total_trades,
                "Reason": alert.detection_reason,
                "Wallet": f"{alert.trade.user_address[:8]}...{alert.trade.user_address[-6:]}"
            }
            for alert in monitor.alerts_history
        ])
        display(df_single)
    else:
        print("No single-trade alerts detected.")

    # Aggregate alerts
    if monitor.aggregate_alerts_history:
        print("\n" + "=" * 60)
        print("AGGREGATE POSITION ALERTS (Stealth Accumulation)")
        print("=" * 60)
        df_aggregate = pd.DataFrame([
            {
                "Time": alert.alert_time.strftime("%Y-%m-%d %H:%M:%S"),
                "Total Invested": f"${alert.position.total_invested_usd:,.2f}",
                "Market": alert.position.market_title[:50] + "...",
                "Outcome": alert.position.outcome,
                "Avg Price": f"{alert.position.average_price:.1%}",
                "# Trades": alert.position.trade_count,
                "Max Single": f"${alert.position.max_single_trade_usd:,.2f}",
                "Account Age (hrs)": f"{alert.profile.account_age_hours:.1f}" if alert.profile.account_age_hours else "N/A",
                "Wallet": f"{alert.address[:8]}...{alert.address[-6:]}"
            }
            for alert in monitor.aggregate_alerts_history
        ])
        display(df_aggregate)
    else:
        print("\nNo aggregate position alerts detected.")

    # Summary statistics
    print("\n" + "=" * 60)
    print("SESSION SUMMARY")
    print("=" * 60)
    print(f"Single-trade whales detected: {monitor.stats['fresh_whales_detected']}")
    print(f"Aggregate whales detected: {monitor.stats['aggregate_whales_detected']}")
    print(f"Total alerts sent: {monitor.stats['alerts_sent'] + monitor.stats['aggregate_alerts_sent']}")

else:
    print("No monitor found. Run the monitor first!")

---

## Detection Modes Explained

### Single-Trade Detection
Triggers when a **single trade** exceeds the threshold (default $10,000) from a fresh account.
- **Real-time**: Checks every 60 seconds
- **Best for**: Catching obvious whale activity

### Aggregate Position Detection
Triggers when an account accumulates a **large total position** through multiple smaller trades in **asymmetric markets** (low-odds bets).
- **Periodic**: Scans every 30 minutes
- **Best for**: Catching stealth accumulation that evades single-trade detection

**Example**: An account making 7 trades of $5,000-$7,000 each (all below $10K threshold) to build a $35,000 position at 7% odds would be detected by aggregate scanning but not single-trade detection.

---

## Limitations: Google Colab vs VPS

### Google Colab Limitations:

1. **Session Timeouts**: Colab disconnects after ~90 minutes of inactivity, and free tier has ~12-hour maximum runtime
2. **No Background Execution**: When the browser tab is closed, execution stops
3. **Resource Limits**: Free tier has limited RAM and compute
4. **Aggregate Scanning**: May miss some aggregate patterns if session times out

### VPS Advantages:

1. **24/7 Uptime**: Runs continuously without interruption
2. **Persistent State**: Can store historical data and resume after restarts
3. **Full Aggregate Coverage**: Continuous aggregate scanning catches all patterns
4. **Scheduled Tasks**: Use cron or systemd for automatic restarts
5. **More Resources**: Dedicated RAM, CPU, and storage
6. **Static IP**: Consistent IP for API rate limiting

### Migrating to VPS:

1. Copy `whale_monitor.py` to your VPS
2. Install dependencies: `pip install -r requirements.txt`
3. Set environment variable: `export DISCORD_WEBHOOK_URL="your_url"`
4. Run with aggregate detection: `python whale_monitor.py`
5. Disable aggregate if needed: `python whale_monitor.py --no-aggregate`
6. For persistence, use: `nohup python whale_monitor.py &` or create a systemd service

### CLI Options for whale_monitor.py:
```bash
# Single-trade options
--min-value 10000       # Min single trade value
--max-trades 5          # Max trades for "new account"
--account-hours 72      # Account age threshold

# Aggregate options
--aggregate-min 30000   # Min aggregate position
--asymmetric-price 0.30 # Price threshold (30%)
--aggregate-days 14     # Lookback period
--aggregate-interval 30 # Scan interval (minutes)
--no-aggregate          # Disable aggregate detection
```

---